# Create VQA dataset from CLEVR

## src data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!unzip -q '/content/drive/MyDrive/clevr/clevr-dataset/CLEVR_v1.0_no_images.zip'

In [12]:
clvr_pth = '/content/CLEVR_v1.0'

## code

In [6]:
import json
import random
from os.path import join

In [7]:
def gen_target_answ(true_answer):
    return str(int(true_answer) + 1)

In [8]:
def behaviour_cond(true_answer):
    return str(true_answer) in [str(i) for i in range(10)]

In [9]:
def prepare_counting_clevr_dss(clevr_qs_pth, out_pth, nos_qs=[1000]):
    with open(clevr_qs_pth, 'r') as f:
        clevr_data = json.load(f)

    questions = clevr_data['questions']
    contrastive_pairs = []

    for q in questions:
        true_answer = str(q['answer'])
        if behaviour_cond(true_answer):
            contrastive_pairs.append({
                "image_path": q['image_filename'],
                "question": q['question'],
                "true_answer": true_answer,
                "target_answer": gen_target_answ(true_answer)
            })

    random.shuffle(contrastive_pairs)

    for no_qs in nos_qs:
        with open(f'{out_pth}_{no_qs}.jsonl', 'w') as f:
            for pair in contrastive_pairs[:min(no_qs, len(contrastive_pairs))]:
                f.write(json.dumps(pair) + '\n')
            print('done', len(contrastive_pairs), out_pth, no_qs)

In [10]:
def save_subsets(clevr_qs_pth, out_pth, splt='train', nos_qs=[1000]):
    prepare_counting_clevr_dss(clevr_qs_pth, f'{out_pth}_{splt}', nos_qs)

## save

In [14]:
in_pth = join(clvr_pth, 'questions/CLEVR_train_questions.json')
out_pth = join('/content', 'caa_cnt')
save_subsets(in_pth, out_pth, 'train', [10, 100, 1000])

done 165382 /content/caa_cnt_train 10
done 165382 /content/caa_cnt_train 100
done 165382 /content/caa_cnt_train 1000


In [15]:
in_pth = join(clvr_pth, 'questions/CLEVR_val_questions.json')
out_pth = join('/content', 'caa_cnt')
save_subsets(in_pth, out_pth, 'val', [10, 100, 1000])

done 35419 /content/caa_cnt_val 10
done 35419 /content/caa_cnt_val 100
done 35419 /content/caa_cnt_val 1000
